# **Baseera — Voice-Guided Object Finder**
NTI Graudation Project

by:  Hager Ali, Mariam Hazzaa, Menna Sobhe, Mariam Mohey


### **Flow**


**Voice input → language detection → LLM extracts target object → multi-model YOLO detection → direction/distance math → LLM composes spoken answer (in the user's language) → text-to-speech**



---



## **What this notebook is, in the context of the larger project**

This is the **object-finder module** of a larger navigation assistant for blind users — the flow that
answers a direct question like *"where is my laptop?"*, as opposed to the continuous background narration

that describes the surroundings unprompted. It's a single-shot request/response interaction: the user asks, the system looks, the system answers.

Everything here runs **locally** (no cloud API calls) — Whisper for speech-to-text, a small local instruct LLM for language understanding, YOLO for detection, and gTTS for speech synthesis.

This keeps it self-contained and easy to run in Colab, at the cost of the response quality you'd get from a larger
cloud LLM. Swapping in a cloud LLM for object extraction / response composition later is a drop-in change to the two functions that call `llm_pipeline`.



---



## **What's new in this version**

| **Feature** | **How it works** |
|---|---|
| **Arabic support** | The Whisper model used here (`base`) is multilingual by default — passing an Arabic recording needs no extra setup, and transcription returns the detected language automatically. |
| **Language-matched responses** | Object *extraction* always normalizes to English internally (YOLO's class names are English, so matching needs a consistent language) — but the final *spoken response* is generated in whichever language the user asked in. English question → English answer. Arabic question → Arabic answer. |
| **Picture-testing mode** | Alongside the live Colab camera capture, you can now point the pipeline at a static image file — useful for testing without a webcam, or for batch-testing against a folder of sample photos. |
| **Multi-model detection** | More than one YOLO checkpoint runs on the same frame; whichever model reports the highest confidence for the target object is used as the answer. This trades some speed for better accuracy on borderline detections. |

## **Two ways to run each stage**

Every stage below has both a **live** path (record audio / capture a photo through the browser — Colab only) and a **test** path (pass a `.wav`/`.mp3` file, a typed string, or an image file directly). Use the test path for development and debugging; the live path for an actual end-to-end demo.


## **Install dependencies**

| Library | Purpose |
|---|---|
| `faster-whisper` | Local, multilingual speech-to-text |
| `ultralytics` | YOLO object detection models |
| `transformers` + `torch` | Local instruct LLM for language understanding / response generation |
| `gTTS` | Text-to-speech, supports both English and Arabic |
| `opencv-python-headless` | Image loading/decoding |
| `langdetect` | Lightweight language guess for *typed* test queries (audio queries get their language from Whisper directly, so this is only a fallback for the text-only test path) |


In [ ]:
!pip install faster-whisper ultralytics transformers gTTS opencv-python-headless torch langdetect -q


## **Imports**

In [ ]:
import os
import re
import base64
from base64 import b64decode

import cv2
import numpy as np
import torch
from gtts import gTTS
from IPython.display import display, Audio
from ultralytics import YOLO
from faster_whisper import WhisperModel
from transformers import pipeline
from langdetect import detect as detect_text_language

try:
    from google.colab.output import eval_js
    from IPython.display import Javascript
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Running in Colab:" if IN_COLAB else "Not running in Colab -- use test-mode cells only.", IN_COLAB)


Running in Colab: True


## **Configuration**

### **Distance estimation**
Distance is estimated with the classic pinhole-camera relationship using each object's *known real-world
width* and the box's *pixel width* in the frame:

```
distance_cm = (real_width_cm * FOCAL_LENGTH) / pixel_width
```

`FOCAL_LENGTH` here is a calibration constant for a typical laptop/phone webcam — **recalibrate this for
your actual camera** by measuring a known object at a known distance and solving for focal length. This
is a coarse estimate, not a precise measurement; treat the resulting distance as a rough bucket, not a
guarantee.


---


### Language settings
`SUPPORTED_LANGUAGES` maps a detected language code to its full name (used in LLM prompts) and its gTTS
code (used for speech synthesis).


In [ ]:
FOCAL_LENGTH = 600  # recalibrate for actual camera -- note above

KNOWN_WIDTHS_CM = {
    "laptop": 35, "cell phone": 7, "person": 50, "chair": 50,
    "bottle": 7, "cup": 9, "book": 20, "keyboard": 45,
    "mouse": 12, "tv": 120, "backpack": 30, "handbag": 25,
    "remote": 5, "umbrella": 100, "couch": 180,
    "default": 30,
}

SUPPORTED_LANGUAGES = {
    "en": {"name": "English", "gtts_code": "en"},
    "ar": {"name": "Arabic", "gtts_code": "ar"},
}
DEFAULT_LANGUAGE = "en"

# Small bilingual dictionaries used by the guaranteed-correct template fallback
# (see Section 9) -- keeps responses understandable even if the LLM's Arabic
# generation goes wrong, since these are hand-written, not model output.
OBJECT_NAME_AR = {
    "laptop": "الحاسوب المحمول", "cell phone": "الهاتف", "person": "شخص",
    "chair": "الكرسي", "bottle": "الزجاجة", "cup": "الكوب", "book": "الكتاب",
    "keyboard": "لوحة المفاتيح", "mouse": "الفأرة", "tv": "التلفاز",
    "backpack": "حقيبة الظهر", "handbag": "الحقيبة", "remote": "جهاز التحكم",
    "umbrella": "المظلة", "couch": "الأريكة",
}
DIRECTION_AR = {
    "left": "اليسار", "right": "اليمين", "center": "الأمام مباشرة",
    "top": "الأعلى", "bottom": "الأسفل",
    "top-left": "أعلى اليسار", "top-right": "أعلى اليمين",
    "bottom-left": "أسفل اليسار", "bottom-right": "أسفل اليمين",
}

# More than one checkpoint runs on every frame (Section 8); the highest-confidence
# match across all of them is used as the final answer. Add/remove entries here.
DETECTION_MODEL_NAMES = {
    "yolov8n": "yolov8n.pt",   # fastest, baseline accuracy
    "yolov8s": "yolov8s.pt",   # slower, generally more accurate on smaller/partial objects
}


## Load models

- **Whisper (`base`, not `base.en`)** — the plain (non-English-suffixed) Whisper checkpoints are
  multilingual. This is what makes Arabic input work with zero extra configuration: `whisper_model.transcribe()`
  auto-detects the spoken language and returns it alongside the transcription.
- **YOLO detection models** — every checkpoint listed in `DETECTION_MODEL_NAMES` is loaded up front so the
  ensemble step (Section 8) can query all of them without a reload per request.
- **Local instruct LLM (Qwen2.5-1.5B-Instruct)** — handles two language-understanding tasks: extracting
  the target object from the spoken query (Section 7), and composing the final spoken response in the
  matching language (Section 9). It's small enough to run without a GPU, at the cost of being less fluent
  than a larger model, especially in Arabic — the template fallback in Section 9 exists precisely to catch
  that.


In [ ]:
# Speech-to-text:
                  # multilingual (English + Arabic + more) out of the box
whisper_model = WhisperModel("base", device="cuda" if torch.cuda.is_available() else "cpu")

# Object detection:
                  # one model object per checkpoint, used together in the ensemble step
detection_models = {name: YOLO(path) for name, path in DETECTION_MODEL_NAMES.items()}

# Local instruct LLM for object extraction + response composition
device_id = 0 if torch.cuda.is_available() else -1
llm_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=device_id,
)

print("Detection models:", list(detection_models.keys()))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Detection models: ['yolov8n', 'yolov8s']


## Live input: browser mic & camera bridges (Colab only)

These two functions use in-browser JavaScript to record audio and capture a photo through Colab's
notebook UI. They **only work in Google Colab**, and only when run interactively (a human needs to click
the on-screen button). Skip this section entirely if you're using the picture/text test-mode instead
(Section 6) — the rest of the pipeline doesn't care which input path produced the audio file or frame.


In [ ]:
def record_audio_colab(filename='query.wav'):
    '''Records a mic clip via the browser. Colab only. Returns the saved file path.'''
    if not IN_COLAB:
        raise RuntimeError("record_audio_colab() only works in Google Colab.")
    js = Javascript('''
        async function recordAudio() {
            const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
            const mediaRecorder = new MediaRecorder(stream);
            let audioChunks = [];
            mediaRecorder.ondataavailable = event => audioChunks.push(event.data);
            mediaRecorder.start();

            const div = document.createElement('div');
            const button = document.createElement('button');
            button.textContent = 'Stop Recording';
            button.style.padding = '10px 20px';
            document.body.appendChild(div);
            div.appendChild(button);

            await new Promise(resolve => button.onclick = resolve);
            mediaRecorder.stop();
            stream.getTracks().forEach(track => track.stop());
            div.remove();

            return new Promise(resolve => {
                mediaRecorder.onstop = async () => {
                    const audioBlob = new Blob(audioChunks, { type: 'audio/wav' });
                    const reader = new FileReader();
                    reader.readAsDataURL(audioBlob);
                    reader.onloadend = () => resolve(reader.result);
                };
            });
        }
    ''')
    display(js)
    data = eval_js('recordAudio()')
    binary = base64.b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

def take_photo_colab_fixed(filename='photo.jpg'):
  js = Javascript("""
    async function takePhoto() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.maxWidth = '100%';

      let stream;
      // Primary attempt: Request environmental camera (phone back camera)
      try {
        stream = await navigator.mediaDevices.getUserMedia({
          video: { facingMode: 'environment' }
        });
      } catch (e) {
        // Fallback attempt: Standard default camera capture
        try {
          stream = await navigator.mediaDevices.getUserMedia({ video: true });
        } catch (err) {
          alert("Camera access denied or unavailable: " + err);
          return null;
        }
      }

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      const button = document.createElement('button');
      button.textContent = '📸 Capture Frame';
      button.style.padding = '10px 20px';
      button.style.margin = '10px 0';
      div.appendChild(button);

      await new Promise((resolve) => button.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.8);
    }
  """)
  display(js)
  data = eval_js('takePhoto()')

  if not data:
    raise RuntimeError('Camera access failed or was denied.')

  binary = b64decode(data.split(',')[1])
  nparr = np.frombuffer(binary, np.uint8)
  frame = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
  return frame

## Test input: picture and text-query mode

For development, debugging, or running outside Colab, skip the live browser capture entirely:

- **Picture testing**: point `load_test_image()` at any `.jpg`/`.png` file instead of using the webcam.
- **Text-query testing**: pass a typed string directly instead of recording audio — this skips Whisper
  entirely and uses `langdetect` to guess the language instead (Whisper's own language detection is more
  reliable, so prefer the audio path when you actually want to test language detection quality).


In [ ]:
def load_test_image(path):
    '''Loads a static image file as a BGR numpy frame -- the same format take_photo_colab() returns,
    so it's a drop-in substitute anywhere a 'frame' is expected below.'''
    frame = cv2.imread(path)
    if frame is None:
        raise FileNotFoundError(f"Could not read image at: {path}")
    return frame


def guess_text_language(text):
    '''Best-effort language guess for a typed (non-audio) query. Falls back to DEFAULT_LANGUAGE
    if detection fails (e.g. very short strings) or returns something we don't support.'''
    try:
        lang = detect_text_language(text)
    except Exception:
        return DEFAULT_LANGUAGE
    return lang if lang in SUPPORTED_LANGUAGES else DEFAULT_LANGUAGE


## Step 1 — Extract the target object (language-aware)

The LLM's job here is narrow: read the query (in English or Arabic) and return **one object name, in
English**, matching common object vocabulary (laptop, cell phone, backpack, bottle, cup, book, etc.).
English output is required regardless of input language, because the detection models' class names are
English — this is the normalization step that lets the rest of the pipeline stay language-agnostic.

The user's spoken language itself is tracked separately (from Whisper's detection or `guess_text_language`)
and carried through to the response step, so the *answer* still comes back in whatever language they asked in.


In [ ]:
def extract_object_local(query_text):
    '''Returns a single lowercase English object name extracted/translated from the query,
    regardless of whether the query itself was in English or Arabic.'''
    messages = [
        {"role": "system", "content": (
            "You are an information extraction system. The user is asking about a missing object, "
            "in English or Arabic. Identify the object and return ONLY its name as a single lowercase "
            "English word or short phrase from common object vocabulary (e.g. laptop, cell phone, "
            "backpack, bottle, cup, book, chair, tv, remote, keyboard, mouse, umbrella, couch). "
            "Return only the object name in English, nothing else -- no translation notes, no punctuation."
        )},
        {"role": "user", "content": query_text},
    ]
    outputs = llm_pipeline(messages, max_new_tokens=10)
    result = outputs[0]["generated_text"][-1]["content"].strip().lower()
    return re.sub(r'[^a-z0-9 ]', '', result)


## Step 2 — Multi-model detection + direction/distance math

Every model in `detection_models` runs on the same frame. Each match against the target object is scored
by that model's own confidence; **the highest-confidence match across all models wins** and is used as the
final answer. This is a simple but effective ensemble strategy — it costs extra compute (running N models
instead of one) in exchange for not being stuck with one model's mistake on a borderline detection.

Direction is computed as a 3×3 grid position (left/center/right combined with top/middle/bottom, e.g.
`"top-left"`) based on where the box center falls relative to the frame, with a 20% dead zone around center
in each axis so small offsets don't get over-classified as a corner. Distance uses the pinhole-camera
formula from Section 3.


In [ ]:
def _box_direction(cx, cy, frame_w, frame_h):
    frame_cx, frame_cy = frame_w // 2, frame_h // 2
    h_dir = "left" if cx < frame_cx - (frame_w * 0.2) else "right" if cx > frame_cx + (frame_w * 0.2) else "center"
    v_dir = "top" if cy < frame_cy - (frame_h * 0.2) else "bottom" if cy > frame_cy + (frame_h * 0.2) else "middle"
    if h_dir != "center" and v_dir != "middle":
        return f"{v_dir}-{h_dir}"
    return h_dir if v_dir == "middle" else v_dir


def analyze_detections_multi_model(frame, target_object, conf=0.3):
    '''
    Runs every loaded detection model against the frame, filters for matches against
    target_object, and returns all matches sorted by confidence (best first) with the
    source model name attached -- so you can see which model 'won' as well as get the
    single best answer via matches[0].
    '''
    h, w = frame.shape[:2]
    all_matches = []

    for model_name, model in detection_models.items():
        results = model(frame, conf=conf, verbose=False)[0]
        for box in results.boxes:
            class_name = model.names[int(box.cls[0])].lower()
            confidence = float(box.conf[0])

            if target_object not in class_name and class_name not in target_object:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            pixel_width = x2 - x1
            cx, cy = x1 + pixel_width // 2, y1 + (y2 - y1) // 2

            real_w_cm = KNOWN_WIDTHS_CM.get(class_name, KNOWN_WIDTHS_CM["default"])
            dist_cm = (real_w_cm * FOCAL_LENGTH) / pixel_width if pixel_width > 0 else 0

            all_matches.append({
                "object": class_name,
                "direction": _box_direction(cx, cy, w, h),
                "distance_m": round(dist_cm / 100, 2),
                "confidence": round(confidence, 2),
                "model": model_name,
            })

    all_matches.sort(key=lambda m: m["confidence"], reverse=True)
    return all_matches


## Step 3 — Compose the spoken response (bilingual, with a safe fallback)

The response is generated by the local LLM **in the same language the user asked in**, using the
best (highest-confidence) match from Section 8. Because a small local model's Arabic output quality can
be inconsistent, every LLM-generated response is checked against a simple script test before being
trusted:

- If `language == "ar"`, the output must actually contain Arabic characters.
- If it doesn't (empty, malformed, or the model answered in English by mistake), fall back to the
  hand-written bilingual template — guaranteed correct, if less natural-sounding.

This guarantees the user always gets an understandable answer in the right language, even when the small
local LLM's generation quality has an off moment.


In [ ]:
_ARABIC_CHAR_RE = re.compile(r'[\u0600-\u06FF]')


def _contains_arabic(text):
    return bool(_ARABIC_CHAR_RE.search(text))


def _template_response(target_object, best_match, language):
    '''Guaranteed-correct fallback using hand-written bilingual templates.'''
    if language == "ar":
        object_ar = OBJECT_NAME_AR.get(target_object, target_object)
        if best_match:
            direction_ar = DIRECTION_AR.get(best_match["direction"], best_match["direction"])
            return (f"نعم، وجدت {object_ar}. إنه في جهة {direction_ar}، "
                    f"على بعد حوالي {best_match['distance_m']} متر.")
        return f"بحثت حولي، لكن لم أتمكن من العثور على {object_ar} في مجال الرؤية."

    # English
    if best_match:
        return (f"Yes, I found your {target_object}. It is located to the {best_match['direction']}, "
                f"about {best_match['distance_m']} meters away.")
    return f"I looked around, but I could not find your {target_object} in the camera view."


def generate_voice_response(target_object, matches, language=DEFAULT_LANGUAGE):
    '''
    matches: output of analyze_detections_multi_model(), already sorted best-first.
    language: 'en' or 'ar' -- the language the user originally asked in.
    '''
    best_match = matches[0] if matches else None
    language_name = SUPPORTED_LANGUAGES.get(language, SUPPORTED_LANGUAGES[DEFAULT_LANGUAGE])["name"]

    facts = (
        f"Object: {target_object}. "
        + (f"Found: yes. Direction: {best_match['direction']}. Distance: {best_match['distance_m']} meters."
           if best_match else "Found: no.")
    )

    messages = [
        {"role": "system", "content": (
            f"You are a voice assistant for a blind user. Given the facts below, write ONE short, "
            f"natural spoken sentence in {language_name} describing whether the object was found and, "
            f"if so, where. Respond ONLY in {language_name}, nothing else."
        )},
        {"role": "user", "content": facts},
    ]
    outputs = llm_pipeline(messages, max_new_tokens=60)
    generated = outputs[0]["generated_text"][-1]["content"].strip()

    if language == "ar" and not _contains_arabic(generated):
        return _template_response(target_object, best_match, language)
    if not generated:
        return _template_response(target_object, best_match, language)
    return generated


## Text-to-speech (bilingual)

gTTS supports both languages used here directly via the `lang` code from `SUPPORTED_LANGUAGES`.

In [ ]:
def speak(text, language=DEFAULT_LANGUAGE, filename="response.mp3"):
    gtts_code = SUPPORTED_LANGUAGES.get(language, SUPPORTED_LANGUAGES[DEFAULT_LANGUAGE])["gtts_code"]
    tts = gTTS(text=text, lang=gtts_code)
    tts.save(filename)
    display(Audio(filename, autoplay=True))
    return filename


## Full pipeline

One function, two modes:

- **`mode="live"`** — records audio and captures a photo through the browser (Colab only).
- **`mode="test"`** — pass `audio_path` OR `query_text` (typed, skips STT), plus `image_path` (skips the
  live camera). Use this for repeatable testing, including testing Arabic input by typing an Arabic string
  directly.


In [ ]:
def run_pipeline(mode="live", audio_path=None, query_text=None, image_path=None, conf=0.3):
    # ---- Step 0: get the query text + detected language ----
    if mode == "live":
        print("Step 1: Speak your question...")
        audio_file = record_audio_colab()
        segments, info = whisper_model.transcribe(audio_file)
        query_text = "".join(segment.text for segment in segments).strip()
        language = info.language if info.language in SUPPORTED_LANGUAGES else DEFAULT_LANGUAGE
    else:
        if audio_path is not None:
            segments, info = whisper_model.transcribe(audio_path)
            query_text = "".join(segment.text for segment in segments).strip()
            language = info.language if info.language in SUPPORTED_LANGUAGES else DEFAULT_LANGUAGE
        elif query_text is not None:
            language = guess_text_language(query_text)
        else:
            raise ValueError("Test mode needs either audio_path or query_text.")

    print(f"   Query ({language}): {query_text!r}")

    # ---- Step 1: extract target object (always normalized to English) ----
    target_object = extract_object_local(query_text)
    print(f"Step 2: Target object -> {target_object!r}")

    # ---- Step 2: get a frame ----
    if mode == "live":
        print("Step 3: Capture frame...")
        frame = take_photo_colab()
    else:
        if image_path is None:
            raise ValueError("Test mode needs image_path.")
        frame = load_test_image(image_path)

    # ---- Step 3: multi-model detection ----
    print("Step 4: Running multi-model detection...")
    matches = analyze_detections_multi_model(frame, target_object, conf=conf)
    if matches:
        print(f"   Best match: {matches[0]}")
        if len(matches) > 1:
            print(f"   ({len(matches)} total matches across models -- see `matches` for all of them)")

    # ---- Step 4: compose + speak response ----
    print("Step 5: Composing response...")
    reply_text = generate_voice_response(target_object, matches, language=language)
    print(f"\nAssistant ({language}): {reply_text}")
    speak(reply_text, language=language)

    return {
        "query_text": query_text,
        "language": language,
        "target_object": target_object,
        "matches": matches,
        "reply_text": reply_text,
    }


## Demo: live run (Colab only)

Speak your question, then capture a photo when prompted.

In [ ]:
if IN_COLAB:
    run_pipeline(mode="live")
else:
    print("Not in Colab -- skip this cell and use the test-mode demo below instead.")


Step 1: Speak your question...


<IPython.core.display.Javascript object>

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Query (en): 'Where is my bottle?'
Step 2: Target object -> 'bottle'
Step 3: Capture frame...


<IPython.core.display.Javascript object>

Step 4: Running multi-model detection...


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Best match: {'object': 'bottle', 'direction': 'center', 'distance_m': 0.33, 'confidence': 0.88, 'model': 'yolov8s'}
   (2 total matches across models -- see `matches` for all of them)
Step 5: Composing response...

Assistant (en): I found a bottle at the center of my desk, just 0.33 meters away.


## Demo: test mode — picture + typed query, English and Arabic

Point `IMAGE_PATH` at any test photo. Try both an English and an Arabic query against the same image to
confirm the language-matching behavior end to end.


In [ ]:
IMAGE_PATH = "/content/images (1).jfif"  # <-- set this to a real test image

run_pipeline(mode="test", query_text="أين الهاتف؟", image_path=IMAGE_PATH)

[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Query (ar): 'أين الهاتف؟'
Step 2: Target object -> 'phone'
Step 4: Running multi-model detection...


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Best match: {'object': 'cell phone', 'direction': 'bottom-left', 'distance_m': 0.22, 'confidence': 0.67, 'model': 'yolov8s'}
Step 5: Composing response...

Assistant (ar): وجدت الهاتف من الأسفل إلى اليسار بمسافة 0.22 متر.


{'query_text': 'أين الهاتف؟',
 'language': 'ar',
 'target_object': 'phone',
 'matches': [{'object': 'cell phone',
   'direction': 'bottom-left',
   'distance_m': 0.22,
   'confidence': 0.67,
   'model': 'yolov8s'}],
 'reply_text': 'وجدت الهاتف من الأسفل إلى اليسار بمسافة 0.22 متر.'}

In [ ]:
run_pipeline(mode="test", query_text="Where is my laptop?", image_path=IMAGE_PATH)


[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Query (en): 'Where is my laptop?'
Step 2: Target object -> 'laptop'
Step 4: Running multi-model detection...


[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


   Best match: {'object': 'laptop', 'direction': 'center', 'distance_m': 0.9, 'confidence': 0.86, 'model': 'yolov8n'}
   (2 total matches across models -- see `matches` for all of them)
Step 5: Composing response...

Assistant (en): The laptop was found at the center of the room, approximately 0.9 meters away from its initial position.


{'query_text': 'Where is my laptop?',
 'language': 'en',
 'target_object': 'laptop',
 'matches': [{'object': 'laptop',
   'direction': 'center',
   'distance_m': 0.9,
   'confidence': 0.86,
   'model': 'yolov8n'},
  {'object': 'laptop',
   'direction': 'center',
   'distance_m': 0.91,
   'confidence': 0.83,
   'model': 'yolov8s'}],
 'reply_text': 'The laptop was found at the center of the room, approximately 0.9 meters away from its initial position.'}

## 14. Notes, limitations & next steps

- **Distance accuracy**: the pinhole-width method here is a coarse estimate — recalibrate `FOCAL_LENGTH`
  for your actual camera, and expect more error on partially occluded or angled objects.
- **Object vocabulary**: matching is against YOLO's COCO classes (80 everyday objects). Anything outside
  that list — most of the custom outdoor/architectural classes from the broader project (pole, curb,
  facade, tuk-tuk, etc.) — needs the separate open-vocabulary detection path described elsewhere in the
  project's architecture guide, not this notebook.
- **Small local LLM quality**: Qwen2.5-1.5B-Instruct is lightweight enough to run without a GPU, but its
  Arabic fluency is noticeably weaker than a larger model's — this is exactly why Section 9 has a
  guaranteed-correct template fallback rather than trusting the LLM output unconditionally. Swapping in a
  larger multilingual model (or a cloud LLM API) would likely improve response naturalness in Arabic.
- **Ensemble strategy**: this notebook picks the single highest-confidence detection across models. A more
  sophisticated ensemble (e.g. weighted box fusion across models before picking a direction/distance)
  would be a reasonable next step if single-model mistakes are still common in practice.
- **This is a single-shot prototype**: it runs one photo/detection per question. The full project design
  splits this into a fast "check recent memory first" cache lookup before falling back to an active scan
  like this notebook does every time — see the object-finder section of the main architecture guide for
  that distinction.
